In [0]:
%sql
MERGE INTO workspace.taxi_medallion.silver_trips t
USING (SELECT trip_id, fare_amount + 100 AS fare_amount
       FROM workspace.taxi_medallion.silver_trips ORDER BY trip_id LIMIT 5) s
ON t.trip_id = s.trip_id
WHEN MATCHED THEN UPDATE SET t.fare_amount = s.fare_amount

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
%sql
DESCRIBE HISTORY workspace.taxi_medallion.silver_trips

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-11-25T04:01:27.187Z,77192597487201,jeffsp_95@hotmail.com,MERGE,"Map(predicate -> [""(trip_id#12327 = trip_id#12355)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [], clusteringOnWriteStatus -> null)",null,List(2012321347126948),d465febb-6e09-4faa-8f8f-b40620228ee3,1125-034651-yz1u4bh6-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 5506, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 6214, materializeSourceTimeMs -> 1020, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2388, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2687)",null,Databricks-Runtime/19.16.x-aarch64-photon-scala2.13
2,2026-11-25T03:57:31.760Z,77192597487201,jeffsp_95@hotmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-04b439cb-03c5-4224-8c80-bc451dcfd25f"",""delta.enableInCommitTimestamps"":""true"",""databricks.internal.autoUpgrades.delta.enableInCommitTimestamps"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-40caca4e-e8ae-4ae2-9922-fd262538e388""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)","List(65855126552279, New Job 2026-11-24 21:54:30, 450728067463967, 183216648136168, 77192597487201, manual, 29272783376030)",List(2012321347126946),66d92dd1-933d-4f89-bc57-1dfde789d1ca,1125-035553-6200vgvp-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1140494, numDeletionVectorsRemoved -> 0, numOutputRows -> 21847, numOutputBytes -> 1140495)",null,Databricks-Runtime/19.16.x-aarch64-photon-scala2.13
1,2026-11-25T03:56:38.416Z,77192597487201,jeffsp_95@hotmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-04b439cb-03c5-4224-8c80-bc451dcfd25f"",""delta.enableInCommitTimestamps"":""true"",""databricks.internal.autoUpgrades.delta.enableInCommitTimestamps"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-40caca4e-e8ae-4ae2-9922-fd262538e388""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)","List(65855126552279, New Job 2026-11-24 21:54:30, 773057593026092, 275319159333913, 77192597487201, manual, 2927278

In [0]:
%sql
SELECT trip_id, fare_amount FROM workspace.taxi_medallion.silver_trips VERSION AS OF 2
ORDER BY trip_id LIMIT 5

trip_id,fare_amount
00063910f5a462dcde0a036a24559ea7896184d3abad676289ba0c2843d8381b,34.0
00086fbefb1c277ef4ce1d2a236e481de31abf81be24ad3fbcd71ed7b805342b,4.0
000b820b05f6a476cf933ade1db8ed67f4aca6f735066c24b0bcb3354527b669,5.0
00127a9c8af6ace278dfd37a4d28c5b7a8d122304f3869d993ac57e0a206e5ac,6.5
00128599dd055cab0fe191e670ae32cabbaca82d6325b687b7e8b3098b27d675,7.0


In [0]:
from pyspark.sql import functions as F
try:
    bad = spark.table("workspace.taxi_medallion.silver_trips").limit(3).withColumn("extra_col", F.lit("x"))
    bad.write.format("delta").mode("append").saveAsTable("workspace.taxi_medallion.silver_trips")
except Exception as e:
    print("BLOCKED:", str(e)[:300])

BLOCKED: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: 8278f5d4-5247-4541-8499-89ad9b0def00).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("merge


In [0]:
%sql
RESTORE TABLE workspace.taxi_medallion.silver_trips TO VERSION AS OF 2

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
1140495,1,2,1,1146001,1140495


In [0]:
%sql
OPTIMIZE workspace.taxi_medallion.silver_trips

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1795579590387, 1795579593025, 8, 0, null, List(0, 0), null, 14, 14, 0, 0, null, null, 0)"
